# Adding Transmission to the Energy System Model

In the previous notebook, we [added a source](../_02_add_component/_1_add_source.ipynb) to our ESM, which defines the basic structure of the energy system such as locations, commodities, and the temporal resolution.

In this notebook, we introduce **transmission components**. Transmission components can **transmit a commodity between locations of the energy system**.

Typical examples of transmission can include:

- transport of electricity through power lines...
- transport of gases per pipelines
- transport of commodities per trucks

## Load ESM

We first load the ESM from the [previous notebook](../_02_add_component/_2_add_sink.ipynb).

In [1]:
import fine as fn
import fine.IOManagement.xarrayIO as xrIO
import pandas as pd
import numpy as np
from pathlib import Path
cwd = Path.cwd().resolve()
base_path = cwd.parents[2]
nc_file = base_path / "examples" / "Examples" / "NetCDF" / "esm_source_sink.nc"

esM = xrIO.readNetCDFtoEnergySystemModel(nc_file)

## Add Transmission

### AC Cables

We can now add AC Cables as a first transmission component. Below you can find a more detailed explanation of the parameters used here.

In [2]:

esM.add(
    fn.Transmission(
        esM = esM,
        name = "AC cables", # Name of the transmission
        commodity = "electricity", # Name of the commodity produced by the source, as set during initialization
        hasCapacityVariable = True, # Specifies whether the source has a capacity variable
        capacityFix = pd.DataFrame(
            [[0, 30], [30, 0]], columns=["regionN", "regionS"], index=["regionN", "regionS"]
            ), # Indicates the fixed capacity of this transmission for each location
        distances = pd.DataFrame(
            [[0, 400], [400, 0]], columns=["regionN", "regionS"], index=["regionN", "regionS"]
            ), # Indicates the distance between the locations for each location
        losses = 0.0001, # Indicates the losses per unit of distance for the transmission
        )
    )

### Natural gas pipelines

In [3]:

esM.add(
    fn.Transmission(
        esM = esM,
        name = "NG pipelines",
        commodity = "naturalGas",
        hasCapacityVariable = True,

        capacityFix = pd.DataFrame(
            [[0, 100], [100, 0]],
            columns=["regionN", "regionS"],
            index=["regionN", "regionS"]
        ),

        distances = pd.DataFrame(
            [[0, 400], [400, 0]],
            columns=["regionN", "regionS"],
            index=["regionN", "regionS"]
        ),

        losses = 0.00002,
    )
)


## Save the Energy System Model

In [4]:
cwd = Path.cwd().resolve()
base_path = cwd.parents[2]
nc_file = base_path / "examples" / "Examples" / "NetCDF" / "esm_source_sink_transmission.nc"

xrIO.writeEnergySystemModelToNetCDF(
    esM, outputFilePath=nc_file, overwriteExisting=True
)


Writing output to netCDF... 
Done. (0.4590 sec)


## General structure of a Transmission Component

The following code snippet shows the structure of the `Transmission` class and its most important arguments.

```python
fn.Transmission(
    esM,
    name,
    commodity,
    losses=0,
    distances=None,
    hasCapacityVariable=True,
    capacityVariableDomain="continuous",
    capacityPerPlantUnit=1,
    hasIsBuiltBinaryVariable=False,
    bigM=None,
    operationRateMax=None,
    operationRateFix=None,
    tsaWeight=1,
    locationalEligibility=None,
    capacityMin=None,
    capacityMax=None,
    partLoadMin=None,
    sharedPotentialID=None,
    linkedQuantityID=None,
    capacityFix=None,
    commissioningMin=None,
    commissioningMax=None,
    commissioningFix=None,
    isBuiltFix=None,
    investPerCapacity=0,
    investIfBuilt=0,
    opexPerOperation=0,
    opexPerCapacity=0,
    opexIfBuilt=0,
    QPcostScale=0,
    interestRate=0.08,
    economicLifetime=10,
    technicalLifetime=None,
    floorTechnicalLifetime=True,
    balanceLimitID=None,
    pathwayBalanceLimitID=None,
    stockCommissioning=None,
    pwlcfParameters=None,
)
```
In the following sections, we explain the most important arguments of a Transmission component.


## Required Arguments

### commodity

`commodity` defines which commodity should be transported.

The commodity must be one of the commodities that were defined when initializing the `EnergySystemModel`.

Typical examples of commodities that can be transported can include:

- electricity 
- hydrogen 
- natural gas...

## Default Arguments




### losses

`losses` defines relative losses per lengthUnit (lengthUnit as specified in the energy system model), in percentage of the commodity flow. This loss factor can capture simple linear losses. 
It is given as a positive float between 0 and 1, or or Pandas DataFrame with positive values (0 <= float <= 1). The row and column indices of the DataFrame have to equal the in the energy system model specified locations.

### distances

`distances` defines distances between locations given in the lengthUnit (lengthUnit as specified in the energy system model). 

It is given as positive float ($\geq 0$) or Pandas DataFrame with positive values ($\geq 0$). The row and column indices of the DataFrame have to equal the in the energy system model specified locations.

### OperationRateMax and OperationRatefix

The parameters `OperationRateMax` and `OperationRatefix` indicate a maximum operation rate and a fixed operation rate, respectively, for all possible connections (both directions) of the transmission component at each time step, if required also for each investment period, by a positive float. 

If hasCapacityVariable is set to True, the values are given relative to the installed capacities (i.e. a value of 1 indicates a utilization of 100% of the capacity). If hasCapacityVariable is set to False, the values are given as absolute values in form of the commodityUnit, referring to the transmitted commodity (before considering losses) during one time step. 

It is given as Pandas DataFrame with positive (>= 0) entries. The row indices have to match the in the energy system model specified time steps. The column indices are combinations of locations (as defined in the energy system model), separated by a underscore (e.g. "location1_location2"). The first location indicates where the commodity is coming from. The second location indicates where the commodity is going too. If a flow is specified from location i to location j, it also has to be specified from j to i. * a dictionary with investment periods as keys and one of the two options above as values.


### opexPerOperation

The parameters `opexPerOperation` describes the cost for one unit of the operation. The cost which is directly proportional to the operation of the component is obtained by multiplying the opexPerOperation parameter with the annual sum of the operational time series of the components. The opexPerOperation can either be given as a float or a Pandas DataFrame with location specific values or a dictionary per investment period with one of the previous options. The cost unit in which the parameter is given has to match the one specified in the energy system model (e.g. Euro, Dollar, 1e6 Euro). The value has to match the unit costUnit/operationUnit (e.g. Euro/kWh, Dollar/kWh). |br| * the default value is 0 :type opexPerOperation: * positive (>=0) float * Pandas DataFrame with positive (>=0).The row and column indices of the DataFrame have to equal the in the energy system model specified locations. * a dictionary with investment periods as keys and one of the two options above as values.


## List of all parameters

Below, after executing the code cell, you will find the list of all parameters of a Transmission Component, along with their description, type, and default value.

In [5]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "NetCDF"))
from docstringTable import display_param_table


display_param_table(fn.Transmission)


,Description,Type,Default
Argument,,,
esM,energy system model to which the component should be added. Used for unit checks.,EnergySystemModel instance from the FINE package,/
name,name of the component. Has to be unique (i.e. no other components with that name can already exist in the EnergySystemModel instance to which the component is added).,string,/
commodity,to the component related commodity.,string,/
losses,"relative losses per lengthUnit (lengthUnit as specified in the energy system model) in percentage of the commodity flow. This loss factor can capture simple linear losses .. math:: trans_{in, ij} = (1 - \\text{losses} \\cdot \\text{distances}) \\cdot trans_{out, ij} (with trans being the commodity flow at a certain point in time and i and j being locations in the energy system). The losses can either be given as a float or a Pandas DataFrame with location specific values.",positive float (0 <= float <= 1) or Pandas DataFrame with positive values (0 <= float <= 1). The row and column indices of the DataFrame have to equal the in the energy system model specified locations.,0
distances,distances between locations given in the lengthUnit (lengthUnit as specified in the energy system model).,positive float (>= 0) or Pandas DataFrame with positive values (>= 0). The row and column indices of the DataFrame have to equal the in the energy system model specified locations.,None
hasCapacityVariable,"specifies if the component should be modeled with a capacity or not. Examples: An electrolyzer has a capacity given in GW_electric -> hasCapacityVariable is True. In the energy system, biogas can, from a model perspective, be converted into methane (and then used in conventional power plants which emit CO2) by getting CO2 from the environment. Thus, using biogas in conventional power plants is, from a balance perspective, CO2 free. This conversion is purely theoretical and does not require a capacity -> hasCapacityVariable is False. A electricity cable has a capacity given in GW_electric -> hasCapacityVariable is True. If the transmission capacity of a component is unlimited -> hasCapacityVariable is False. A wind turbine has a capacity given in GW_electric -> hasCapacityVariable is True. Emitting CO2 into the environment is not per se limited by a capacity -> hasCapacityVariable is False.",boolean,True
capacityVariableDomain,"describes the mathematical domain of the capacity variables, if they are specified. By default, the domain is specified as 'continuous' and thus declares the variables as positive (>=0) real values. The second input option that is available for this parameter is 'discrete', which declares the variables as positive (>=0) integer values.",string ('continuous' or 'discrete'),'continuous'
capacityPerPlantUnit,"capacity of one plant of the component (in the specified physicalUnit of the plant). The default is 1, thus the number of plants is equal to the installed capacity. This parameter should be specified when using a 'discrete' capacityVariableDomain. It can be specified when using a 'continuous' variable domain.",dict of strictly positive float or strictly positive float,1
hasIsBuiltBinaryVariable,"specifies if binary decision variables should be declared for each eligible location of the component, which indicates if the component is built at that location or not (dimension=1dim). each eligible connection of the transmission component, which indicates if the component is built between two locations or not (dimension=2dim). The binary variables can be used to enforce one-time investment cost or capacity-independent annual operation cost. If a minimum capacity is specified and this parameter is set to True, the minimum capacities are only considered if a component is built (i.e. if a component is built at that location, it has to be built with a minimum capacity of XY GW, otherwise it is set to 0 GW).",boolean,False
